In [1]:
"""
PM2.5 Model Performance Visualisations
=======================================
Loads all saved pkl models and produces publication-quality plots.

Plots saved to:
  C:\\Users\\kesha\\OneDrive - dtu.ac.in\\Desktop\\aodtopm25\\reports\\

Run:
    pip install matplotlib seaborn scipy
    python pm25_plots.py
"""

import warnings, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from scipy import stats
from sklearn.metrics import (r2_score, mean_absolute_error,
                             mean_squared_error, mean_absolute_percentage_error)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

# ════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════
DATA_PATH  = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet"
MODEL_DIR  = pathlib.Path(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\src\models")
REPORT_DIR = pathlib.Path(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FRAC = 0.60
VAL_FRAC   = 0.80

# ── Palette (colorblind-safe) ──────────────────────────────────────────────
MODEL_COLORS = {
    "XGBoost"         : "#E63946",
    "LightGBM"        : "#F4A261",
    "RandomForest"    : "#2A9D8F",
    "GradientBoosting": "#457B9D",
    "CatBoost"        : "#6A0572",
    "Ensemble"        : "#1D3557",
}
SPLIT_COLORS = {"train": "#2A9D8F", "val": "#F4A261", "test": "#E63946"}

plt.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : "#F8F9FA",
    "axes.grid"        : True,
    "grid.color"       : "white",
    "grid.linewidth"   : 1.2,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.family"      : "DejaVu Sans",
    "axes.titlesize"   : 13,
    "axes.labelsize"   : 11,
    "xtick.labelsize"  : 9,
    "ytick.labelsize"  : 9,
})

def save(fig, name):
    p = REPORT_DIR / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {p.name}")

# ════════════════════════════════════════════
# 1. DATA + FEATURES  (identical to pipeline)
# ════════════════════════════════════════════
print("=" * 60)
print("  PM2.5 Visualisation Suite")
print("=" * 60)

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["station_name", "date"]).reset_index(drop=True)

grp = df.groupby("station_name")["pm25"]
for lag in [1, 2, 3, 7, 14]:
    df[f"pm25_lag{lag}"] = grp.shift(lag)
for window in [3, 7, 14, 30]:
    df[f"pm25_roll{window}"] = (
        grp.shift(1).transform(lambda x: x.rolling(window, min_periods=1).mean()))
df["pm25_ewm7"]            = grp.shift(1).transform(lambda x: x.ewm(span=7,  min_periods=1).mean())
df["pm25_ewm14"]           = grp.shift(1).transform(lambda x: x.ewm(span=14, min_periods=1).mean())
df["pm25_lag1_diff"]       = df["pm25_lag1"]  - df["pm25_lag2"]
df["pm25_lag7_diff"]       = df["pm25_lag1"]  - df["pm25_lag7"]
df["pm25_roll3_diff"]      = df["pm25_roll3"] - df["pm25_roll7"]
df["pm25_lag1_vs_roll7"]   = df["pm25_lag1"]  - df["pm25_roll7"]
df["pm25_roll3_vs_roll14"] = df["pm25_roll3"] - df["pm25_roll14"]

df["station_enc"] = LabelEncoder().fit_transform(df["station_name"])
if "season" in df.columns:
    df["season_enc"] = LabelEncoder().fit_transform(df["season"].astype(str))

AOD_SENTINEL = ["AOD_mean","AOD_max","AOD_p75","AOD_lag1","AOD_lag2","AOD_roll3","AOD_roll7"]
for col in AOD_SENTINEL:
    if col in df.columns:
        df[f"{col}_missing"] = (df[col] == -1).astype(int)
        df[col] = df[col].replace(-1, np.nan)

DROP = {"pm25","date","station_name","season","lat","lon"}
FEATURE_COLS = [c for c in df.columns if c not in DROP and df[c].dtype != object]
TARGET = "pm25"
USE_RESID = "station_month_pm" in df.columns
if USE_RESID:
    df["pm25_resid"] = df[TARGET] - df["station_month_pm"]

df = df.sort_values("date").reset_index(drop=True)
n  = len(df)
t1 = int(n * TRAIN_FRAC)
t2 = int(n * VAL_FRAC)

X_tr = df[FEATURE_COLS].iloc[:t1]
X_va = df[FEATURE_COLS].iloc[t1:t2]
X_te = df[FEATURE_COLS].iloc[t2:]
y_raw_tr = df[TARGET].iloc[:t1].values
y_raw_va = df[TARGET].iloc[t1:t2].values
y_raw_te = df[TARGET].iloc[t2:].values

if USE_RESID:
    y_tr = df["pm25_resid"].iloc[:t1]
    y_va = df["pm25_resid"].iloc[t1:t2]
    base_tr = df["station_month_pm"].iloc[:t1].values
    base_va = df["station_month_pm"].iloc[t1:t2].values
    base_te = df["station_month_pm"].iloc[t2:].values

imp_path = MODEL_DIR / "imputer.pkl"
_raw = joblib.load(imp_path) if imp_path.exists() else None
if _raw and hasattr(_raw, "transform") and hasattr(_raw, "statistics_"):
    imp = _raw
else:
    imp = SimpleImputer(strategy="median").fit(X_tr)

X_tr_i = pd.DataFrame(imp.transform(X_tr), columns=FEATURE_COLS)
X_va_i = pd.DataFrame(imp.transform(X_va), columns=FEATURE_COLS)
X_te_i = pd.DataFrame(imp.transform(X_te), columns=FEATURE_COLS)

# ── Load models ────────────────────────────────────────────────────────────
PKL = {"XGBoost":"xgboost.pkl","LightGBM":"lightgbm.pkl",
       "RandomForest":"randomforest.pkl","GradientBoosting":"gradientboosting.pkl",
       "CatBoost":"catboost.pkl"}

models = {}
for name, fname in PKL.items():
    p = MODEL_DIR / fname
    if p.exists():
        models[name] = joblib.load(p)
        print(f"  Loaded : {name}")

def predict(model, X, base=None):
    p = model.predict(X)
    if USE_RESID and base is not None:
        p = p + base
    return np.maximum(p, 0)

def metrics(yt, yp):
    return dict(R2   = r2_score(yt, yp),
                MAE  = mean_absolute_error(yt, yp),
                RMSE = mean_squared_error(yt, yp) ** 0.5,
                MAPE = mean_absolute_percentage_error(yt, yp) * 100)

# ── Collect all predictions & metrics ──────────────────────────────────────
all_preds  = {}
all_metrics = {}

for name, model in models.items():
    tr_pred = predict(model, X_tr_i, base_tr if USE_RESID else None)
    va_pred = predict(model, X_va_i, base_va if USE_RESID else None)
    te_pred = predict(model, X_te_i, base_te if USE_RESID else None)
    all_preds[name]   = {"train": tr_pred, "val": va_pred, "test": te_pred}
    all_metrics[name] = {"train": metrics(y_raw_tr, tr_pred),
                         "val"  : metrics(y_raw_va, va_pred),
                         "test" : metrics(y_raw_te, te_pred)}

# ── Blend ensemble ─────────────────────────────────────────────────────────
blend_path = MODEL_DIR / "blend_meta.pkl"
if blend_path.exists():
    meta = joblib.load(blend_path)
    mnames = meta["model_names"]
    bw     = meta["blend_weights"]
    ens_te = np.maximum(
        sum(bw[i] * all_preds[mnames[i]]["test"] for i in range(len(mnames))
            if mnames[i] in all_preds), 0)
    all_preds["Ensemble"]   = {"test": ens_te}
    all_metrics["Ensemble"] = {"test": metrics(y_raw_te, ens_te)}

dates_te = df["date"].iloc[t2:].values
print(f"\n  Generating plots …")

# ════════════════════════════════════════════
# PLOT 1 — Metrics Comparison Bar Chart (R², MAE, RMSE per model × split)
# ════════════════════════════════════════════
splits = ["train", "val", "test"]
metric_names = ["R2", "MAE", "RMSE"]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Model Performance Across Train / Val / Test", fontsize=15, fontweight="bold", y=1.02)

for ax, met in zip(axes, metric_names):
    x      = np.arange(len(models))
    width  = 0.22
    for j, split in enumerate(splits):
        vals = [all_metrics[n][split][met] for n in models]
        bars = ax.bar(x + (j - 1) * width, vals, width,
                      label=split.capitalize(),
                      color=SPLIT_COLORS[split], alpha=0.88,
                      edgecolor="white", linewidth=0.8)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005 * max(vals),
                    f"{v:.2f}", ha="center", va="bottom",
                    fontsize=7.5, fontweight="bold")
    ax.set_title(met, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(list(models.keys()), rotation=25, ha="right")
    ax.legend(fontsize=8)
    if met == "R2":
        ax.set_ylim(0.7, 1.02)
    ax.set_ylabel(met)

plt.tight_layout()
save(fig, "01_metrics_comparison.png")

# ════════════════════════════════════════════
# PLOT 2 — Scatter: Predicted vs Actual (test set, per model)
# ════════════════════════════════════════════
n_models = len(models)
ncols = 3
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = axes.flatten()
fig.suptitle("Predicted vs Actual PM2.5 — Test Set", fontsize=15, fontweight="bold")

for ax, (name, model) in zip(axes, models.items()):
    yp = all_preds[name]["test"]
    yt = y_raw_te
    color = MODEL_COLORS.get(name, "#555")

    ax.scatter(yt, yp, alpha=0.25, s=8, color=color, rasterized=True)

    # Perfect prediction line
    lim = max(yt.max(), yp.max()) * 1.05
    ax.plot([0, lim], [0, lim], "k--", lw=1.2, label="Perfect fit")

    # Regression line
    slope, intercept, r, *_ = stats.linregress(yt, yp)
    xs = np.linspace(0, lim, 200)
    ax.plot(xs, slope * xs + intercept, color=color, lw=2,
            label=f"Fit  y={slope:.2f}x+{intercept:.1f}")

    m = all_metrics[name]["test"]
    ax.set_title(f"{name}\nR²={m['R2']:.4f}  MAE={m['MAE']:.2f}  RMSE={m['RMSE']:.2f}",
                 fontweight="bold", fontsize=10)
    ax.set_xlabel("Actual PM2.5 (µg/m³)")
    ax.set_ylabel("Predicted PM2.5 (µg/m³)")
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.legend(fontsize=8)

for ax in axes[n_models:]:
    ax.set_visible(False)

plt.tight_layout()
save(fig, "02_scatter_predicted_vs_actual.png")

# ════════════════════════════════════════════
# PLOT 3 — Residual Distribution (test set)
# ════════════════════════════════════════════
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
axes = axes.flatten()
fig.suptitle("Residual Distribution — Test Set  (Actual − Predicted)", fontsize=14, fontweight="bold")

for ax, (name, model) in zip(axes, models.items()):
    resid = y_raw_te - all_preds[name]["test"]
    color = MODEL_COLORS.get(name, "#555")
    ax.hist(resid, bins=60, color=color, alpha=0.75, edgecolor="white", linewidth=0.5)
    ax.axvline(0,       color="black", lw=1.5, ls="--", label="Zero error")
    ax.axvline(resid.mean(), color="red", lw=1.5, ls="-",
               label=f"Mean={resid.mean():.2f}")
    ax.set_title(f"{name}  (σ={resid.std():.2f})", fontweight="bold")
    ax.set_xlabel("Residual (µg/m³)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

for ax in axes[n_models:]:
    ax.set_visible(False)

plt.tight_layout()
save(fig, "03_residual_distribution.png")

# ════════════════════════════════════════════
# PLOT 4 — Overfit Gap Heatmap (R² across splits)
# ════════════════════════════════════════════
splits_order = ["train", "val", "test"]
r2_matrix = pd.DataFrame(
    {split: [all_metrics[n][split]["R2"] for n in models]
     for split in splits_order},
    index=list(models.keys())
)
gap_col = r2_matrix["train"] - r2_matrix["test"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                         gridspec_kw={"width_ratios": [3, 1]})
fig.suptitle("Overfitting Analysis — R² Heatmap & Gap", fontsize=14, fontweight="bold")

# Heatmap
sns.heatmap(r2_matrix, ax=axes[0], annot=True, fmt=".4f",
            cmap="RdYlGn", vmin=0.75, vmax=1.0,
            linewidths=1, linecolor="white",
            cbar_kws={"label": "R²"})
axes[0].set_title("R² per Model × Split", fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("")

# Gap bar
colors = ["#E63946" if g >= 0.15 else "#F4A261" if g >= 0.10 else "#2A9D8F"
          for g in gap_col]
bars = axes[1].barh(list(models.keys()), gap_col.values,
                    color=colors, edgecolor="white", linewidth=0.8)
for bar, v in zip(bars, gap_col):
    axes[1].text(v + 0.002, bar.get_y() + bar.get_height() / 2,
                 f"{v:.4f}", va="center", fontsize=9, fontweight="bold")
axes[1].axvline(0.10, color="#F4A261", lw=1.5, ls="--", label="Warn (0.10)")
axes[1].axvline(0.20, color="#E63946", lw=1.5, ls="--", label="Overfit (0.20)")
axes[1].set_title("Gap  (Train R² − Test R²)", fontweight="bold")
axes[1].set_xlabel("Gap")
axes[1].legend(fontsize=8)
axes[1].set_xlim(0, 0.25)

plt.tight_layout()
save(fig, "04_overfit_gap_heatmap.png")

# ════════════════════════════════════════════
# PLOT 5 — Time Series: Actual vs All Models (test period, first 500 pts)
# ════════════════════════════════════════════
N_SHOW = 500
fig, ax = plt.subplots(figsize=(18, 6))
fig.suptitle(f"Time Series — Actual vs Predicted (Test set, first {N_SHOW} samples)",
             fontsize=14, fontweight="bold")

ax.plot(range(N_SHOW), y_raw_te[:N_SHOW], color="black",
        lw=1.8, label="Actual", zorder=10)

for name in models:
    color = MODEL_COLORS.get(name, "#aaa")
    ax.plot(range(N_SHOW), all_preds[name]["test"][:N_SHOW],
            color=color, lw=1.0, alpha=0.75, label=name)

if "Ensemble" in all_preds:
    ax.plot(range(N_SHOW), all_preds["Ensemble"]["test"][:N_SHOW],
            color=MODEL_COLORS["Ensemble"], lw=2.0, ls="--",
            alpha=0.9, label="Ensemble")

ax.set_xlabel("Sample index (test set)")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.legend(fontsize=9, ncol=4, loc="upper right")
plt.tight_layout()
save(fig, "05_timeseries_actual_vs_predicted.png")

# ════════════════════════════════════════════
# PLOT 6 — Feature Importance (XGBoost, Top 25)
# ════════════════════════════════════════════
if "XGBoost" in models:
    xgb_m = models["XGBoost"]
    fi = pd.Series(xgb_m.feature_importances_, index=FEATURE_COLS).nlargest(25)

    fig, ax = plt.subplots(figsize=(10, 9))
    colors_fi = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(fi)))[::-1]
    bars = ax.barh(fi.index[::-1], fi.values[::-1],
                   color=colors_fi, edgecolor="white", linewidth=0.6)
    for bar, v in zip(bars, fi.values[::-1]):
        ax.text(v + 0.0005, bar.get_y() + bar.get_height() / 2,
                f"{v:.4f}", va="center", fontsize=8)
    ax.set_title("XGBoost — Top 25 Feature Importances (Test-set refit)",
                 fontweight="bold", fontsize=13)
    ax.set_xlabel("Importance Score")
    plt.tight_layout()
    save(fig, "06_feature_importance_xgboost.png")

# ════════════════════════════════════════════
# PLOT 7 — Error by PM2.5 Concentration Bin (test)
# ════════════════════════════════════════════
bins   = [0, 30, 60, 100, 150, 200, 300, 600]
labels = ["0–30", "30–60", "60–100", "100–150", "150–200", "200–300", "300+"]
bin_col = pd.cut(y_raw_te, bins=bins, labels=labels, right=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Error by PM2.5 Concentration Bin — Test Set", fontsize=14, fontweight="bold")

# MAE per bin per model
mae_bin = {}
for name in models:
    resid = np.abs(y_raw_te - all_preds[name]["test"])
    mae_bin[name] = pd.Series(resid).groupby(bin_col).mean().values

x = np.arange(len(labels))
width = 0.15
for j, (name, vals) in enumerate(mae_bin.items()):
    offset = (j - len(models) / 2) * width + width / 2
    axes[0].bar(x + offset, vals, width, label=name,
                color=MODEL_COLORS.get(name, "#888"), alpha=0.85,
                edgecolor="white", linewidth=0.6)

axes[0].set_title("MAE by Concentration Bin", fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=20)
axes[0].set_xlabel("PM2.5 Concentration Range (µg/m³)")
axes[0].set_ylabel("MAE (µg/m³)")
axes[0].legend(fontsize=8)

# Sample count per bin (context)
counts = pd.Series(y_raw_te).groupby(bin_col).count()
axes[1].bar(labels, counts.values, color="#457B9D", alpha=0.85,
            edgecolor="white", linewidth=0.8)
for i, (label, c) in enumerate(zip(labels, counts.values)):
    axes[1].text(i, c + 20, str(c), ha="center", fontsize=9, fontweight="bold")
axes[1].set_title("Sample Count per Bin", fontweight="bold")
axes[1].set_xlabel("PM2.5 Concentration Range (µg/m³)")
axes[1].set_ylabel("Number of Samples")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
save(fig, "07_error_by_concentration_bin.png")

# ════════════════════════════════════════════
# PLOT 8 — Radar Chart: All Metrics Normalised
# ════════════════════════════════════════════
from matplotlib.patches import FancyArrowPatch

metric_labels = ["R²", "1−nMAE", "1−nRMSE", "1−nMAPE"]
# normalise to [0,1]: higher = better
te_r2   = np.array([all_metrics[n]["test"]["R2"]   for n in models])
te_mae  = np.array([all_metrics[n]["test"]["MAE"]  for n in models])
te_rmse = np.array([all_metrics[n]["test"]["RMSE"] for n in models])
te_mape = np.array([all_metrics[n]["test"]["MAPE"] for n in models])

def norm_inv(arr):    # lower is better → invert to [0,1]
    mn, mx = arr.min(), arr.max()
    return 1 - (arr - mn) / (mx - mn + 1e-9)

scores = np.column_stack([
    (te_r2  - te_r2.min())  / (te_r2.max()  - te_r2.min()  + 1e-9),
    norm_inv(te_mae),
    norm_inv(te_rmse),
    norm_inv(te_mape),
])

angles = np.linspace(0, 2 * np.pi, len(metric_labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})
fig.suptitle("Model Comparison — Normalised Metrics (Test Set)",
             fontsize=14, fontweight="bold", y=1.01)

for i, name in enumerate(models):
    vals = scores[i].tolist() + scores[i][:1].tolist()
    ax.plot(angles, vals, lw=2, color=MODEL_COLORS.get(name, "#888"), label=name)
    ax.fill(angles, vals, alpha=0.07, color=MODEL_COLORS.get(name, "#888"))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=8)
ax.legend(loc="lower right", bbox_to_anchor=(1.35, -0.05), fontsize=9)
ax.grid(color="white", linewidth=1.2)
ax.set_facecolor("#F0F4F8")

plt.tight_layout()
save(fig, "08_radar_chart_metrics.png")

# ════════════════════════════════════════════
# PLOT 9 — Residual vs Predicted (heteroscedasticity check)
# ════════════════════════════════════════════
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
axes = axes.flatten()
fig.suptitle("Residuals vs Predicted — Heteroscedasticity Check (Test Set)",
             fontsize=14, fontweight="bold")

for ax, (name, model) in zip(axes, models.items()):
    yp    = all_preds[name]["test"]
    resid = y_raw_te - yp
    color = MODEL_COLORS.get(name, "#555")
    ax.scatter(yp, resid, alpha=0.2, s=7, color=color, rasterized=True)
    ax.axhline(0, color="black", lw=1.5, ls="--")
    # Smoothed trend
    sort_idx = np.argsort(yp)
    yp_s, resid_s = yp[sort_idx], resid[sort_idx]
    window = max(len(yp_s) // 30, 10)
    smooth = pd.Series(resid_s).rolling(window, center=True, min_periods=1).mean()
    ax.plot(yp_s, smooth, color="red", lw=2, label="Smoothed trend")
    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("Predicted PM2.5 (µg/m³)")
    ax.set_ylabel("Residual (µg/m³)")
    ax.legend(fontsize=8)

for ax in axes[n_models:]:
    ax.set_visible(False)

plt.tight_layout()
save(fig, "09_residuals_vs_predicted.png")

# ════════════════════════════════════════════
# PLOT 10 — Summary Dashboard (single-page overview)
# ════════════════════════════════════════════
fig = plt.figure(figsize=(20, 14))
fig.suptitle("PM2.5 Model Performance Dashboard", fontsize=18,
             fontweight="bold", y=1.01)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# [0,0] R² bar — test only
ax0 = fig.add_subplot(gs[0, 0])
names_list = list(models.keys())
r2_vals = [all_metrics[n]["test"]["R2"] for n in names_list]
colors  = [MODEL_COLORS.get(n, "#888") for n in names_list]
bars = ax0.bar(names_list, r2_vals, color=colors, edgecolor="white", linewidth=0.8)
for bar, v in zip(bars, r2_vals):
    ax0.text(bar.get_x() + bar.get_width() / 2, v + 0.002,
             f"{v:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax0.set_ylim(0.75, 0.85)
ax0.set_title("Test R²", fontweight="bold")
ax0.set_ylabel("R²")
ax0.tick_params(axis="x", rotation=20)

# [0,1] MAE bar — test only
ax1 = fig.add_subplot(gs[0, 1])
mae_vals = [all_metrics[n]["test"]["MAE"] for n in names_list]
bars = ax1.bar(names_list, mae_vals, color=colors, edgecolor="white", linewidth=0.8)
for bar, v in zip(bars, mae_vals):
    ax1.text(bar.get_x() + bar.get_width() / 2, v + 0.05,
             f"{v:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax1.set_title("Test MAE (µg/m³)", fontweight="bold")
ax1.set_ylabel("MAE")
ax1.tick_params(axis="x", rotation=20)

# [0,2] RMSE bar — test only
ax2 = fig.add_subplot(gs[0, 2])
rmse_vals = [all_metrics[n]["test"]["RMSE"] for n in names_list]
bars = ax2.bar(names_list, rmse_vals, color=colors, edgecolor="white", linewidth=0.8)
for bar, v in zip(bars, rmse_vals):
    ax2.text(bar.get_x() + bar.get_width() / 2, v + 0.05,
             f"{v:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax2.set_title("Test RMSE (µg/m³)", fontweight="bold")
ax2.set_ylabel("RMSE")
ax2.tick_params(axis="x", rotation=20)

# [1,0:2] Time series — best model (CatBoost) + ensemble
ax3 = fig.add_subplot(gs[1, 0:2])
N = 300
ax3.plot(range(N), y_raw_te[:N], color="black", lw=2, label="Actual", zorder=5)
best_name = min(all_metrics, key=lambda n: all_metrics[n]["test"]["MAE"]
                if "test" in all_metrics[n] else 999)
ax3.plot(range(N), all_preds[best_name]["test"][:N],
         color=MODEL_COLORS.get(best_name, "#E63946"),
         lw=1.3, alpha=0.85, label=f"Best model ({best_name})")
if "Ensemble" in all_preds:
    ax3.plot(range(N), all_preds["Ensemble"]["test"][:N],
             color=MODEL_COLORS["Ensemble"], lw=1.5, ls="--",
             alpha=0.9, label="Ensemble")
ax3.set_title(f"Time Series — Actual vs Best Model + Ensemble (first {N} test samples)",
              fontweight="bold")
ax3.set_xlabel("Sample index")
ax3.set_ylabel("PM2.5 (µg/m³)")
ax3.legend(fontsize=9, loc="upper right")

# [1,2] Gap bar
ax4 = fig.add_subplot(gs[1, 2])
gaps  = [all_metrics[n]["train"]["R2"] - all_metrics[n]["test"]["R2"] for n in names_list]
gcols = ["#E63946" if g >= 0.15 else "#F4A261" if g >= 0.10 else "#2A9D8F" for g in gaps]
bars  = ax4.bar(names_list, gaps, color=gcols, edgecolor="white", linewidth=0.8)
for bar, v in zip(bars, gaps):
    ax4.text(bar.get_x() + bar.get_width() / 2, v + 0.002,
             f"{v:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax4.axhline(0.10, color="#F4A261", lw=1.5, ls="--", label="Warn 0.10")
ax4.axhline(0.20, color="#E63946", lw=1.5, ls="--", label="Overfit 0.20")
ax4.set_title("Overfit Gap  (Train R² − Test R²)", fontweight="bold")
ax4.set_ylabel("Gap")
ax4.legend(fontsize=8)
ax4.tick_params(axis="x", rotation=20)

plt.tight_layout()
save(fig, "10_dashboard_summary.png")

# ════════════════════════════════════════════
# DONE
# ════════════════════════════════════════════
print("\n" + "=" * 60)
print(f"  All 10 plots saved to:")
print(f"  {REPORT_DIR}")
print("=" * 60)
print("""
  01_metrics_comparison.png       — R², MAE, RMSE across splits
  02_scatter_predicted_vs_actual.png — Predicted vs Actual scatter
  03_residual_distribution.png    — Residual histograms
  04_overfit_gap_heatmap.png      — Heatmap + gap bar
  05_timeseries_actual_vs_predicted.png — Time series overlay
  06_feature_importance_xgboost.png — Top-25 XGBoost features
  07_error_by_concentration_bin.png — MAE per PM2.5 range
  08_radar_chart_metrics.png      — Normalised radar chart
  09_residuals_vs_predicted.png   — Heteroscedasticity check
  10_dashboard_summary.png        — Single-page overview
""")

  PM2.5 Visualisation Suite
  Loaded : XGBoost
  Loaded : LightGBM
  Loaded : RandomForest
  Loaded : GradientBoosting
  Loaded : CatBoost

  Generating plots …
  Saved → 01_metrics_comparison.png
  Saved → 02_scatter_predicted_vs_actual.png
  Saved → 03_residual_distribution.png
  Saved → 04_overfit_gap_heatmap.png
  Saved → 05_timeseries_actual_vs_predicted.png
  Saved → 06_feature_importance_xgboost.png
  Saved → 07_error_by_concentration_bin.png
  Saved → 08_radar_chart_metrics.png
  Saved → 09_residuals_vs_predicted.png
  Saved → 10_dashboard_summary.png

  All 10 plots saved to:
  C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports

  01_metrics_comparison.png       — R², MAE, RMSE across splits
  02_scatter_predicted_vs_actual.png — Predicted vs Actual scatter
  03_residual_distribution.png    — Residual histograms
  04_overfit_gap_heatmap.png      — Heatmap + gap bar
  05_timeseries_actual_vs_predicted.png — Time series overlay
  06_feature_importance_xgboost.png — To

In [2]:
"""
Feature Comparison Plots — final_ml_featured.parquet
=====================================================
Generates 6 publication-quality plots comparing features
in the dataset.

Saved to:
  C:\\Users\\kesha\\OneDrive - dtu.ac.in\\Desktop\\aodtopm25\\reports\\features\\

Run:
    python feature_plots.py
"""

import warnings, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# ════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════
DATA_PATH  = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet"
REPORT_DIR = pathlib.Path(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\features")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : "#F8F9FA",
    "axes.grid"        : True,
    "grid.color"       : "white",
    "grid.linewidth"   : 1.2,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.family"      : "DejaVu Sans",
    "axes.titlesize"   : 13,
    "axes.labelsize"   : 11,
    "xtick.labelsize"  : 9,
    "ytick.labelsize"  : 9,
})

PALETTE    = ["#E63946","#F4A261","#2A9D8F","#457B9D","#6A0572",
              "#264653","#E9C46A","#A8DADC","#F1FAEE","#1D3557"]
SEASON_PAL = {"Winter":"#457B9D","Pre-Monsoon":"#F4A261",
              "Monsoon":"#2A9D8F","Post-Monsoon":"#E63946"}

def save(fig, name):
    p = REPORT_DIR / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {p.name}")

# ════════════════════════════════════════════
# LOAD
# ════════════════════════════════════════════
print("=" * 62)
print("  Feature Comparison Plots")
print("=" * 62)

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df["year"]  = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")

print(f"  Loaded : {df.shape[0]:,} rows | {df.shape[1]} cols")
print(f"  Saving to: {REPORT_DIR}\n")

# ── Helper: pick columns that actually exist ───────────────────────────────
def pick(candidates, df=df):
    return [c for c in candidates if c in df.columns]

# ── Feature groups ─────────────────────────────────────────────────────────
AOD_COLS  = pick(["AOD_mean","AOD_max","AOD_p75","AOD_lag1","AOD_lag2","AOD_roll3","AOD_roll7"])
MET_COLS  = pick(["SPEED_mean","TLML_mean","TLML_max","moisture_max","rain_3day",
                   "rain_lag1","PRECTOTLAND","RHOA_mean","QV2M_mean","T2M_mean",
                   "WS10M_mean","PS_mean","U10M_mean","V10M_mean"])
STAT_COLS = pick(["station_pm_mean","station_pm_std","station_month_pm"])
LAG_COLS  = pick(["pm25_lag1","pm25_lag2","pm25_lag3","pm25_lag7","pm25_lag14"])
ROLL_COLS = pick(["pm25_roll3","pm25_roll7","pm25_roll14","pm25_roll30"])
EWM_COLS  = pick(["pm25_ewm7","pm25_ewm14"])

MONTH_ORDER = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

# ════════════════════════════════════════════
# PLOT 1 — AOD vs PM2.5 Scatter Grid
#           One panel per AOD feature, coloured by season
# ════════════════════════════════════════════
print("[1] AOD vs PM2.5 scatter grid …")

aod_plot = AOD_COLS[:6] if len(AOD_COLS) >= 6 else AOD_COLS
ncols    = 3
nrows    = (len(aod_plot) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5 * nrows))
fig.suptitle("AOD Features vs PM2.5 — Correlation Scatter (coloured by Season)",
             fontsize=15, fontweight="bold")
axes = axes.flatten()

for ax, col in zip(axes, aod_plot):
    tmp = df[["pm25", col]].replace(-1, np.nan).dropna()
    if "season" in df.columns:
        tmp["season"] = df.loc[tmp.index, "season"]
        for s, grp in tmp.groupby("season"):
            ax.scatter(grp[col], grp["pm25"],
                       alpha=0.18, s=6, label=s,
                       color=SEASON_PAL.get(s, "#888"), rasterized=True)
        ax.legend(fontsize=7, markerscale=3, framealpha=0.7)
    else:
        ax.scatter(tmp[col], tmp["pm25"], alpha=0.2, s=6,
                   color="#457B9D", rasterized=True)

    # Regression line
    if len(tmp) > 10:
        slope, intercept, r, p, _ = stats.linregress(tmp[col], tmp["pm25"])
        xs = np.linspace(tmp[col].min(), tmp[col].max(), 200)
        ax.plot(xs, slope * xs + intercept, color="#E63946", lw=2)
        ax.set_title(f"{col}\n r = {r:.3f}  (p {'< 0.001' if p < 0.001 else f'= {p:.3f}'})",
                     fontweight="bold", fontsize=10)

    ax.set_xlabel(col)
    ax.set_ylabel("PM2.5 (µg/m³)")

for ax in axes[len(aod_plot):]:
    ax.set_visible(False)

plt.tight_layout()
save(fig, "F01_aod_vs_pm25_scatter.png")

# ════════════════════════════════════════════
# PLOT 2 — PM2.5 Distribution by Month & Season
#           Box plots + violin
# ════════════════════════════════════════════
print("[2] PM2.5 seasonal & monthly distribution …")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("PM2.5 Distribution by Month and Season", fontsize=15, fontweight="bold")

# Monthly box plot
month_order_present = [m for m in MONTH_ORDER if m in df["month_name"].values]
sns.boxplot(data=df, x="month_name", y="pm25",
            order=month_order_present,
            palette=sns.color_palette("coolwarm", 12),
            width=0.6, linewidth=0.8,
            flierprops=dict(marker="o", ms=2, alpha=0.3),
            ax=axes[0])
axes[0].set_title("Monthly PM2.5 Distribution", fontweight="bold")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("PM2.5 (µg/m³)")
axes[0].tick_params(axis="x", rotation=0)

# Monthly median line overlay
monthly_med = df.groupby("month_name")["pm25"].median().reindex(month_order_present)
ax_twin = axes[0].twinx()
ax_twin.plot(range(len(monthly_med)), monthly_med.values,
             "k-o", lw=2, ms=5, label="Median")
ax_twin.set_ylabel("Median PM2.5", color="black")
ax_twin.spines["top"].set_visible(False)

# Seasonal violin — drawn manually to avoid seaborn palette bugs in newer versions
if "season" in df.columns:
    season_order = [s for s in ["Winter","Pre-Monsoon","Monsoon","Post-Monsoon"]
                    if s in df["season"].values]
    for i, s in enumerate(season_order):
        sub = df.loc[df["season"] == s, "pm25"].dropna().values
        if len(sub) < 10:
            continue
        parts = axes[1].violinplot(sub, positions=[i], widths=0.6,
                                   showmedians=True, showextrema=True)
        col = SEASON_PAL.get(s, "#888")
        for pc in parts.get("bodies", []):
            pc.set_facecolor(col)
            pc.set_alpha(0.75)
        for key in ["cmedians", "cmins", "cmaxes", "cbars"]:
            if key in parts:
                parts[key].set_color(col)
                parts[key].set_linewidth(1.5)
    axes[1].set_xticks(range(len(season_order)))
    axes[1].set_xticklabels(season_order)
    axes[1].set_title("Seasonal PM2.5 Distribution (Violin)", fontweight="bold")
    axes[1].set_xlabel("Season")
    axes[1].set_ylabel("PM2.5 (µg/m³)")
else:
    sns.boxplot(data=df, x="year", y="pm25", ax=axes[1],
                palette="coolwarm")
    axes[1].set_title("PM2.5 Distribution by Year", fontweight="bold")

plt.tight_layout()
save(fig, "F02_pm25_seasonal_monthly.png")

# ════════════════════════════════════════════
# PLOT 3 — Correlation Heatmap
#           PM2.5 + AOD + Top Met features
# ════════════════════════════════════════════
print("[3] Feature correlation heatmap …")

corr_cols = pick(["pm25"] + AOD_COLS[:5] + MET_COLS[:8] + STAT_COLS[:3])
corr_df   = df[corr_cols].replace(-1, np.nan).copy()

# Drop cols with >60% missing
corr_df   = corr_df.loc[:, corr_df.isnull().mean() < 0.6]
corr_mat  = corr_df.corr()

mask = np.triu(np.ones_like(corr_mat, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 12))
fig.suptitle("Feature Correlation Matrix (Pearson)", fontsize=15, fontweight="bold")

sns.heatmap(corr_mat, mask=mask, ax=ax,
            cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 7.5},
            linewidths=0.5, linecolor="white",
            cbar_kws={"label": "Pearson r", "shrink": 0.8})

ax.set_title("Lower triangle: Pearson r — red = positive, blue = negative",
             fontsize=10, style="italic", pad=6)
plt.tight_layout()
save(fig, "F03_feature_correlation_heatmap.png")

# ════════════════════════════════════════════
# PLOT 4 — Meteorological Features vs PM2.5
#           2-row grid: scatter + regression per met variable
# ════════════════════════════════════════════
print("[4] Meteorological features vs PM2.5 …")

met_plot = MET_COLS[:8]
if not met_plot:
    print("  [SKIP] No met columns found.")
else:
    ncols = 4
    nrows = (len(met_plot) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    fig.suptitle("Meteorological Features vs PM2.5", fontsize=15, fontweight="bold")
    axes = axes.flatten()

    for ax, col in zip(axes, met_plot):
        tmp = df[["pm25", col]].dropna()
        # Hex bin — handles large N better than scatter
        hb = ax.hexbin(tmp[col], tmp["pm25"],
                       gridsize=35, cmap="YlOrRd",
                       mincnt=1, linewidths=0.2)
        plt.colorbar(hb, ax=ax, label="Count", pad=0.02)

        if len(tmp) > 10:
            slope, intercept, r, p, _ = stats.linregress(tmp[col], tmp["pm25"])
            xs = np.linspace(tmp[col].min(), tmp[col].max(), 200)
            ax.plot(xs, slope * xs + intercept, "b-", lw=1.8)
            ax.set_title(f"{col}\n r = {r:.3f}", fontweight="bold", fontsize=10)

        ax.set_xlabel(col)
        ax.set_ylabel("PM2.5 (µg/m³)")

    for ax in axes[len(met_plot):]:
        ax.set_visible(False)

    plt.tight_layout()
    save(fig, "F04_met_features_vs_pm25.png")

# ════════════════════════════════════════════
# PLOT 5 — PM2.5 Lag & Rolling Feature Importance
#           Correlation of lag/roll features with pm25
#           + autocorrelation structure
# ════════════════════════════════════════════
print("[5] Lag & rolling feature correlation …")

# Build lag cols if not already present
grp = df.groupby("station_name")["pm25"]
for lag in [1, 2, 3, 7, 14]:
    c = f"pm25_lag{lag}"
    if c not in df.columns:
        df[c] = grp.shift(lag)
for w in [3, 7, 14, 30]:
    c = f"pm25_roll{w}"
    if c not in df.columns:
        df[c] = grp.shift(1).transform(lambda x: x.rolling(w, min_periods=1).mean())
for sp, c in [(7,"pm25_ewm7"),(14,"pm25_ewm14")]:
    if c not in df.columns:
        df[c] = grp.shift(1).transform(lambda x: x.ewm(span=sp, min_periods=1).mean())

lag_roll_cols = pick(
    ["pm25_lag1","pm25_lag2","pm25_lag3","pm25_lag7","pm25_lag14",
     "pm25_roll3","pm25_roll7","pm25_roll14","pm25_roll30",
     "pm25_ewm7","pm25_ewm14"]
)

corr_vals = {c: df[["pm25", c]].dropna().corr().iloc[0, 1]
             for c in lag_roll_cols}
corr_s = pd.Series(corr_vals).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("PM2.5 Temporal Features — Autocorrelation Analysis",
             fontsize=15, fontweight="bold")

# Bar chart — correlation with pm25
colors_bar = ["#E63946" if "lag" in c else "#2A9D8F" if "roll" in c else "#F4A261"
               for c in corr_s.index]
bars = axes[0].barh(corr_s.index[::-1], corr_s.values[::-1],
                    color=colors_bar[::-1], edgecolor="white", linewidth=0.7)
for bar, v in zip(bars, corr_s.values[::-1]):
    axes[0].text(v + 0.003, bar.get_y() + bar.get_height() / 2,
                 f"{v:.3f}", va="center", fontsize=9)
axes[0].set_xlim(0, 1.05)
axes[0].set_title("Pearson r with PM2.5\n(red=lag | green=rolling | orange=EWM)",
                  fontweight="bold")
axes[0].set_xlabel("Correlation coefficient")

# Line plot — lag decay curve
lag_nums  = [1, 2, 3, 7, 14]
lag_corrs = [corr_vals.get(f"pm25_lag{l}", np.nan) for l in lag_nums]
roll_nums  = [3, 7, 14, 30]
roll_corrs = [corr_vals.get(f"pm25_roll{w}", np.nan) for w in roll_nums]

axes[1].plot(lag_nums,  lag_corrs,  "o-", color="#E63946", lw=2, ms=7, label="Lag")
axes[1].plot(roll_nums, roll_corrs, "s-", color="#2A9D8F", lw=2, ms=7, label="Rolling mean")
for sp, col, mk in [(7,"#F4A261","^"),(14,"#6A0572","D")]:
    v = corr_vals.get(f"pm25_ewm{sp}", np.nan)
    if not np.isnan(v):
        axes[1].scatter([sp], [v], color=col, marker=mk, s=100, zorder=5,
                        label=f"EWM-{sp}")
axes[1].set_title("Autocorrelation Decay Curve", fontweight="bold")
axes[1].set_xlabel("Lag / Window size (days)")
axes[1].set_ylabel("Pearson r with PM2.5")
axes[1].set_ylim(0.5, 1.0)
axes[1].legend(fontsize=9)

plt.tight_layout()
save(fig, "F05_lag_rolling_autocorrelation.png")

# ════════════════════════════════════════════
# PLOT 6 — Station-level Feature Comparison
#           Per-station PM2.5 mean, AOD mean, met mean
#           Sorted by PM2.5 mean, top-20 stations
# ════════════════════════════════════════════
print("[6] Station-level feature comparison …")

stat_agg = df.groupby("station_name").agg(
    pm25_mean   = ("pm25",    "mean"),
    pm25_std    = ("pm25",    "std"),
    pm25_median = ("pm25",    "median"),
    count       = ("pm25",    "count"),
).reset_index()

# AOD mean if available
if "AOD_mean" in df.columns:
    aod_agg = df.replace({"AOD_mean": {-1: np.nan}}) \
                .groupby("station_name")["AOD_mean"].mean().reset_index()
    aod_agg.columns = ["station_name", "aod_mean"]
    stat_agg = stat_agg.merge(aod_agg, on="station_name", how="left")

# Met feature if available
met_feat = next((c for c in ["SPEED_mean","TLML_mean","moisture_max"] if c in df.columns), None)
if met_feat:
    met_agg = df.groupby("station_name")[met_feat].mean().reset_index()
    met_agg.columns = ["station_name", "met_val"]
    stat_agg = stat_agg.merge(met_agg, on="station_name", how="left")

stat_agg = stat_agg.sort_values("pm25_mean", ascending=False).head(20).reset_index(drop=True)
labels   = stat_agg["station_name"].str[:14]   # truncate long names

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle("Station-Level Feature Comparison — Top 20 Stations by PM2.5",
             fontsize=15, fontweight="bold")

# [0,0] PM2.5 mean + std error bars
ax0 = fig.add_subplot(gs[0, 0])
colors_st = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(stat_agg)))
bars = ax0.bar(range(len(stat_agg)), stat_agg["pm25_mean"],
               yerr=stat_agg["pm25_std"], capsize=3,
               color=colors_st, edgecolor="white", linewidth=0.6,
               error_kw={"elinewidth": 1, "ecolor": "#555"})
ax0.set_xticks(range(len(stat_agg)))
ax0.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax0.set_title("Mean PM2.5 ± Std Dev per Station", fontweight="bold")
ax0.set_ylabel("PM2.5 (µg/m³)")

# [0,1] AOD mean per station
ax1 = fig.add_subplot(gs[0, 1])
if "aod_mean" in stat_agg.columns:
    ax1.bar(range(len(stat_agg)), stat_agg["aod_mean"],
            color="#457B9D", edgecolor="white", linewidth=0.6, alpha=0.85)
    ax1.set_xticks(range(len(stat_agg)))
    ax1.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax1.set_title("Mean AOD per Station", fontweight="bold")
    ax1.set_ylabel("AOD")
else:
    ax1.text(0.5, 0.5, "AOD_mean not available", ha="center", va="center",
             transform=ax1.transAxes, fontsize=13, color="gray")
    ax1.set_title("AOD (not available)", fontweight="bold")

# [1,0] PM2.5 median per station
ax2 = fig.add_subplot(gs[1, 0])
ax2.bar(range(len(stat_agg)), stat_agg["pm25_median"],
        color="#2A9D8F", edgecolor="white", linewidth=0.6, alpha=0.85)
ax2.set_xticks(range(len(stat_agg)))
ax2.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax2.set_title("Median PM2.5 per Station", fontweight="bold")
ax2.set_ylabel("PM2.5 (µg/m³)")

# [1,1] PM2.5 vs AOD dual-axis, or met variable
ax3 = fig.add_subplot(gs[1, 1])
x = range(len(stat_agg))
ax3.bar(x, stat_agg["pm25_mean"], color="#E63946", alpha=0.7,
        label="PM2.5 mean", edgecolor="white", linewidth=0.6)
ax3.set_ylabel("PM2.5 (µg/m³)", color="#E63946")
ax3.tick_params(axis="y", labelcolor="#E63946")

if "aod_mean" in stat_agg.columns:
    ax3b = ax3.twinx()
    ax3b.plot(x, stat_agg["aod_mean"], "o-",
              color="#457B9D", lw=2, ms=5, label="AOD mean")
    ax3b.set_ylabel("AOD", color="#457B9D")
    ax3b.tick_params(axis="y", labelcolor="#457B9D")
    ax3b.spines["top"].set_visible(False)
    lines1, labs1 = ax3.get_legend_handles_labels()
    lines2, labs2 = ax3b.get_legend_handles_labels()
    ax3.legend(lines1 + lines2, labs1 + labs2, fontsize=8, loc="upper right")
elif "met_val" in stat_agg.columns:
    ax3b = ax3.twinx()
    ax3b.plot(x, stat_agg["met_val"], "s-",
              color="#2A9D8F", lw=2, ms=5, label=met_feat)
    ax3b.set_ylabel(met_feat, color="#2A9D8F")
    ax3b.tick_params(axis="y", labelcolor="#2A9D8F")
    ax3b.spines["top"].set_visible(False)

ax3.set_xticks(x)
ax3.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax3.set_title("PM2.5 vs AOD — Dual Axis per Station", fontweight="bold")

plt.tight_layout()
save(fig, "F06_station_feature_comparison.png")

# ════════════════════════════════════════════
# DONE
# ════════════════════════════════════════════
print("\n" + "=" * 62)
print(f"  All 6 plots saved to:")
print(f"  {REPORT_DIR}")
print("=" * 62)
print("""
  F01_aod_vs_pm25_scatter.png         — AOD features vs PM2.5
  F02_pm25_seasonal_monthly.png       — Monthly & seasonal distribution
  F03_feature_correlation_heatmap.png — Full correlation matrix
  F04_met_features_vs_pm25.png        — Meteorological features vs PM2.5
  F05_lag_rolling_autocorrelation.png — Lag/roll autocorrelation decay
  F06_station_feature_comparison.png  — Station-level PM2.5 + AOD
""")

  Feature Comparison Plots
  Loaded : 66,617 rows | 64 cols
  Saving to: C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\features

[1] AOD vs PM2.5 scatter grid …
  Saved → F01_aod_vs_pm25_scatter.png
[2] PM2.5 seasonal & monthly distribution …
  Saved → F02_pm25_seasonal_monthly.png
[3] Feature correlation heatmap …
  Saved → F03_feature_correlation_heatmap.png
[4] Meteorological features vs PM2.5 …
  Saved → F04_met_features_vs_pm25.png
[5] Lag & rolling feature correlation …
  Saved → F05_lag_rolling_autocorrelation.png
[6] Station-level feature comparison …
  Saved → F06_station_feature_comparison.png

  All 6 plots saved to:
  C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\features

  F01_aod_vs_pm25_scatter.png         — AOD features vs PM2.5
  F02_pm25_seasonal_monthly.png       — Monthly & seasonal distribution
  F03_feature_correlation_heatmap.png — Full correlation matrix
  F04_met_features_vs_pm25.png        — Meteorological features vs PM2.5
  F

In [11]:
"""
Comprehensive Model Analysis — PM2.5 AOD Study
===============================================
Tests overfitting, compares all models, and visualises:

  PLOT 01 — Overfitting diagnosis  (train vs test R² / MAE / RMSE)
  PLOT 02 — Predicted vs Actual scatter  (all models)
  PLOT 03 — Time-series comparison  (first 500 test samples)
  PLOT 04 — Residual distributions
  PLOT 05 — MAE by PM2.5 concentration bin
  PLOT 06 — Residuals by season
  PLOT 07 — Top-20 feature importances  (all models)
  PLOT 08 — Feature correlation heatmap  (top-25 features)
  PLOT 09 — PM2.5 vs key features  (scatter + regression)
  PLOT 10 — Ensemble weight breakdown + metric summary table
  PLOT 11 — Clean metric bar chart overview

Saved to:
  C:\\Users\\kesha\\OneDrive - dtu.ac.in\\Desktop\\aodtopm25\\reports\\withoutlag\\

Run:
    python model_analysis_plots.py
"""

import warnings, pathlib, os
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot   as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.metrics       import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
DATA_PATH  = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet"
MODEL_DIR  = pathlib.Path(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\models")
REPORT_DIR = pathlib.Path(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\reports\withoutlag")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_DATE = pd.Timestamp("2024-07-01")

C = {
    "RandomForest"    : "#2A9D8F",
    "XGBoost"         : "#E63946",
    "LightGBM"        : "#F4A261",
    "GradientBoosting": "#457B9D",
    "Ensemble"        : "#1D3557",
}
SEASON_PAL = {
    "Winter"      : "#1D6FA4",
    "Pre-Monsoon" : "#F4A261",
    "Monsoon"     : "#2A9D8F",
    "Post-Monsoon": "#E76F51",
}

plt.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : "#F0F4F8",
    "axes.grid"        : True,
    "grid.color"       : "white",
    "grid.linewidth"   : 1.4,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.family"      : "DejaVu Sans",
    "axes.titlesize"   : 12,
    "axes.labelsize"   : 10,
    "xtick.labelsize"  : 8.5,
    "ytick.labelsize"  : 8.5,
    "legend.framealpha": 0.9,
    "legend.fontsize"  : 8.5,
})

def save(fig, name, dpi=160):
    p = REPORT_DIR / name
    fig.savefig(p, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  ✓  {name}")

def metrics(y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = float(np.mean(y_pred - y_true))
    return dict(R2=r2, MAE=mae, RMSE=rmse, Bias=bias)


# ═══════════════════════════════════════════════════════════
# 1. LOAD DATA
# ═══════════════════════════════════════════════════════════
print("=" * 60)
print("  Loading data ...")
print("=" * 60)

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
print(f"  Shape : {df.shape}")

# keep original season strings for plot 06 BEFORE encoding
df_season_str = df["season"].copy() if "season" in df.columns else None

# encode season to numeric so models can consume it
if "season" in df.columns:
    df["season"] = LabelEncoder().fit_transform(df["season"].astype(str))

# columns to exclude — lat / lon / season are KEPT because
# the saved models were trained with them
EXCLUDE = {
    "pm25", "date", "station_name", "station_id",
    "latitude", "longitude",
    "aod_quality_cat", "station_month_aod_n",
}

feature_cols = [
    c for c in df.columns
    if c not in EXCLUDE
    and df[c].dtype in ["float64", "float32", "int64", "int32", "bool", "uint8"]
]
print(f"  Features : {len(feature_cols)}")

# temporal split
train_df = df[df["date"] <  SPLIT_DATE].copy()
test_df  = df[df["date"] >= SPLIT_DATE].copy()

X_train = train_df[feature_cols].astype(float)
y_train = train_df["pm25"].values
X_test  = test_df[feature_cols].astype(float)
y_test  = test_df["pm25"].values

print(f"  Train : {X_train.shape}  |  Test : {X_test.shape}")


# ═══════════════════════════════════════════════════════════
# 2. LOAD MODELS
# ═══════════════════════════════════════════════════════════
print("\n  Loading models ...")

PKL = {
    "RandomForest"    : "rf_model.pkl",
    "XGBoost"         : "xgb_model.pkl",
    "LightGBM"        : "lgb_model.pkl",
    "GradientBoosting": "gb_model.pkl",
}
models = {}
for name, fname in PKL.items():
    p = MODEL_DIR / fname
    if p.exists():
        models[name] = joblib.load(p)
        print(f"    Loaded : {name}")
    else:
        print(f"    MISSING: {fname} — skipping")

if not models:
    raise RuntimeError("No models found in MODEL_DIR. Check path.")

# align column order to what the models expect
first_model = list(models.values())[0]
if hasattr(first_model, "feature_names_in_"):
    model_feat_cols = list(first_model.feature_names_in_)
    for c in model_feat_cols:
        if c not in X_train.columns:
            print(f"    Adding missing column '{c}' filled with 0")
            X_train[c] = 0.0
            X_test[c]  = 0.0
    X_train      = X_train[model_feat_cols]
    X_test       = X_test[model_feat_cols]
    feature_cols = model_feat_cols
    print(f"    Column order aligned  ({len(feature_cols)} features)")

# predictions
tr_pred, te_pred = {}, {}
for name, m in models.items():
    tr_pred[name] = np.maximum(m.predict(X_train), 0)
    te_pred[name] = np.maximum(m.predict(X_test),  0)

# ensemble
ens_w_path  = MODEL_DIR / "ensemble_weights.pkl"
ens_weights = joblib.load(ens_w_path) if ens_w_path.exists() else None

if ens_weights:
    key_map = {"rf":"RandomForest", "xgb":"XGBoost",
               "lgb":"LightGBM",    "gb":"GradientBoosting"}
    ens_te = np.zeros(len(y_test))
    ens_tr = np.zeros(len(y_train))
    for k, w in ens_weights.items():
        nm = key_map.get(k, k)
        if nm in te_pred:
            ens_te += w * te_pred[nm]
            ens_tr += w * tr_pred[nm]
    te_pred["Ensemble"] = np.maximum(ens_te, 0)
    tr_pred["Ensemble"] = np.maximum(ens_tr, 0)
    print("    Ensemble predictions computed")

model_names = list(te_pred.keys())

# metrics
tr_metrics = {n: metrics(y_train, tr_pred[n]) for n in model_names}
te_metrics = {n: metrics(y_test,  te_pred[n]) for n in model_names}

print("\n  OVERFITTING CHECK")
print(f"  {'Model':<22} {'Train R2':>9} {'Test R2':>9} {'Delta R2':>10}  "
      f"{'Train MAE':>10} {'Test MAE':>9} {'Delta MAE':>10}")
print(f"  {'-'*85}")
for n in model_names:
    tr, te   = tr_metrics[n], te_metrics[n]
    gap_r2   = tr["R2"]  - te["R2"]
    gap_mae  = te["MAE"] - tr["MAE"]
    flag     = "  << OVERFIT" if gap_r2 > 0.08 else ""
    print(f"  {n:<22} {tr['R2']:>9.4f} {te['R2']:>9.4f} {gap_r2:>10.4f}  "
          f"{tr['MAE']:>10.2f} {te['MAE']:>9.2f} {gap_mae:>10.2f}{flag}")


# ═══════════════════════════════════════════════════════════
# PLOT 01 — Overfitting Diagnosis
# ═══════════════════════════════════════════════════════════
print("\n[Plot 01] Overfitting diagnosis ...")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Plot 01 — Overfitting Diagnosis: Train vs Test Performance",
             fontsize=14, fontweight="bold", color="#1D3557", y=1.02)

for ax, (met, label) in zip(axes, [
    ("R2",   "R2  (higher = better)"),
    ("MAE",  "MAE ug/m3  (lower = better)"),
    ("RMSE", "RMSE ug/m3  (lower = better)"),
]):
    names = [n for n in model_names if n != "Ensemble"]
    tr_v  = [tr_metrics[n][met] for n in names]
    te_v  = [te_metrics[n][met] for n in names]
    x, w  = np.arange(len(names)), 0.32

    ax.bar(x - w/2, tr_v, w, label="Train", color="#2A9D8F", alpha=0.85, edgecolor="white")
    ax.bar(x + w/2, te_v, w, label="Test",  color="#E63946", alpha=0.85, edgecolor="white")

    rng = max(max(tr_v), max(te_v)) - min(min(tr_v), min(te_v))
    rng = rng if rng > 0 else 1
    for i, (tv, ev) in enumerate(zip(tr_v, te_v)):
        gap = tv - ev if met == "R2" else ev - tv
        col = "#C0392B" if abs(gap) > (0.08 if met == "R2" else 5) else "#2C3E50"
        ax.text(x[i], max(tv, ev) + rng * 0.03,
                f"D{abs(gap):.3f}", ha="center", fontsize=7.5,
                color=col, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=18, ha="right")
    ax.set_ylabel(label)
    ax.set_title(met, fontweight="bold", color="#1D3557")
    ax.legend()
    if met == "R2":
        ax.set_ylim(min(min(tr_v), min(te_v)) * 0.96, 1.01)

plt.tight_layout()
save(fig, "plot01_overfitting_diagnosis.png")


# ═══════════════════════════════════════════════════════════
# PLOT 02 — Predicted vs Actual Scatter
# ═══════════════════════════════════════════════════════════
print("[Plot 02] Predicted vs Actual scatter ...")

ncols = 3
nrows = (len(model_names) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5.5*nrows))
axes = np.array(axes).flatten()
fig.suptitle("Plot 02 — Predicted vs Actual (Test Set)",
             fontsize=14, fontweight="bold", color="#1D3557", y=1.01)

for i, name in enumerate(model_names):
    ax  = axes[i]
    yp  = te_pred[name]
    col = C.get(name, "#888")
    lim = max(float(y_test.max()), float(yp.max())) * 1.04

    ax.plot([0, lim], [0, lim], "k--", lw=1.6, label="Perfect fit", zorder=10)
    ax.fill_between([0, lim], [-15, lim-15], [15, lim+15], alpha=0.07, color=col)
    ax.scatter(y_test, yp, alpha=0.15, s=6, color=col, rasterized=True)

    sl, ic, r, *_ = stats.linregress(y_test, yp)
    xs = np.linspace(0, lim, 300)
    ax.plot(xs, sl*xs + ic, color=col, lw=2.0, alpha=0.9, label=f"Fit  r={r:.3f}")

    m = te_metrics[name]
    ax.text(0.04, 0.95,
            f"R2={m['R2']:.3f}\nMAE={m['MAE']:.1f}\nRMSE={m['RMSE']:.1f}\nBias={m['Bias']:.1f}",
            transform=ax.transAxes, fontsize=8, va="top",
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec=col, alpha=0.9))

    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel("Actual PM2.5 (ug/m3)")
    ax.set_ylabel("Predicted PM2.5 (ug/m3)")
    ax.set_title(name, fontweight="bold", color="#1D3557")
    ax.legend(fontsize=7.5)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
save(fig, "plot02_predicted_vs_actual.png")


# ═══════════════════════════════════════════════════════════
# PLOT 03 — Time Series Comparison
# ═══════════════════════════════════════════════════════════
print("[Plot 03] Time-series comparison ...")

N = min(500, len(y_test))
fig, ax = plt.subplots(figsize=(22, 6))
fig.suptitle("Plot 03 — Time Series: Actual vs All Models (first 500 test samples)",
             fontsize=13, fontweight="bold", color="#1D3557")

ax.plot(range(N), y_test[:N], color="black", lw=2.2, label="Actual", zorder=10)
styles = ["-", "--", "-.", ":", (0, (3, 1, 1, 1))]
for i, name in enumerate(model_names):
    ax.plot(range(N), te_pred[name][:N],
            color=C.get(name, "#888"),
            lw=2.2 if name == "Ensemble" else 1.5,
            ls=styles[i % len(styles)],
            alpha=0.85, label=name, zorder=5+i)

ax.set_xlabel("Sample index (test set)")
ax.set_ylabel("PM2.5 (ug/m3)")
ax.legend(ncol=3, fontsize=8.5, loc="upper right")
ax.set_xlim(0, N - 1)
plt.tight_layout()
save(fig, "plot03_timeseries_comparison.png")


# ═══════════════════════════════════════════════════════════
# PLOT 04 — Residual Distributions
# ═══════════════════════════════════════════════════════════
print("[Plot 04] Residual distributions ...")

ncols = 3
nrows = (len(model_names) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5.5*nrows))
axes = np.array(axes).flatten()
fig.suptitle("Plot 04 — Residual Distributions (Test Set)",
             fontsize=14, fontweight="bold", color="#1D3557", y=1.01)

for i, name in enumerate(model_names):
    ax    = axes[i]
    resid = y_test - te_pred[name]
    col   = C.get(name, "#888")

    ax.hist(resid, bins=60, color=col, alpha=0.75, edgecolor="white", density=True)
    xr = np.linspace(float(resid.min()), float(resid.max()), 300)
    ax.plot(xr, stats.gaussian_kde(resid)(xr), color="black", lw=2.0)
    ax.axvline(0,            color="red",  lw=1.8, ls="--", label="Zero")
    ax.axvline(resid.mean(), color="navy", lw=1.5, ls=":",
               label=f"mean={resid.mean():.1f}")

    ax.set_xlabel("Residual (Actual - Predicted) ug/m3")
    ax.set_ylabel("Density")
    ax.set_title(name, fontweight="bold", color="#1D3557")
    ax.legend(fontsize=8)

    _, p_sw = stats.shapiro(resid[:min(3000, len(resid))])
    ax.text(0.97, 0.95, f"Shapiro p={p_sw:.3f}",
            transform=ax.transAxes, fontsize=7.5, ha="right", va="top", color="#555")

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
save(fig, "plot04_residual_distributions.png")


# ═══════════════════════════════════════════════════════════
# PLOT 05 — MAE by PM2.5 Concentration Bin
# ═══════════════════════════════════════════════════════════
print("[Plot 05] MAE by concentration bin ...")

bins_e   = [0, 30, 60, 100, 150, 200, 300, 700]
bin_labs = ["0-30", "30-60", "60-100", "100-150", "150-200", "200-300", "300+"]
bin_col  = pd.cut(y_test, bins=bins_e, labels=bin_labs, right=False)
counts_b = [(bin_col == lb).sum() for lb in bin_labs]

fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle("Plot 05 — MAE by PM2.5 Concentration Bin",
             fontsize=13, fontweight="bold", color="#1D3557")

x = np.arange(len(bin_labs))
w = 0.80 / len(model_names)
for j, name in enumerate(model_names):
    mae_bins = [
        np.mean(np.abs(y_test[bin_col == lb] - te_pred[name][bin_col == lb]))
        if (bin_col == lb).sum() > 0 else 0
        for lb in bin_labs
    ]
    offset = (j - len(model_names) / 2) * w + w / 2
    ax.bar(x + offset, mae_bins, w,
           label=name, color=C.get(name, "#888"),
           alpha=0.85, edgecolor="white", linewidth=0.4)

ax2 = ax.twinx()
ax2.plot(x, counts_b, "ko-", lw=1.8, ms=5, label="Sample count")
ax2.set_ylabel("Sample count", fontsize=9)
ax2.spines["top"].set_visible(False)

ax.set_xticks(x)
ax.set_xticklabels(bin_labs, rotation=20, ha="right")
ax.set_xlabel("PM2.5 Concentration Range (ug/m3)")
ax.set_ylabel("MAE (ug/m3)")
l1, lb1 = ax.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1+l2, lb1+lb2, fontsize=8, ncol=3, loc="upper left")
plt.tight_layout()
save(fig, "plot05_mae_by_pm25_bin.png")


# ═══════════════════════════════════════════════════════════
# PLOT 06 — Residuals by Season
# ═══════════════════════════════════════════════════════════
# print("[Plot 06] Residuals by season ...")

# if df_season_str is not None:
#     test_season_str = df_season_str[df["date"] >= SPLIT_DATE].values
#     season_order    = ["Winter", "Pre-Monsoon", "Monsoon", "Post-Monsoon"]

#     ncols = 3
#     nrows = (len(model_names) + ncols - 1) // ncols
#     fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5.5*nrows))
#     axes = np.array(axes).flatten()
#     fig.suptitle("Plot 06 — Residuals by Season (Test Set)",
#                  fontsize=14, fontweight="bold", color="#1D3557", y=1.01)

#     for idx, name in enumerate(model_names):
#         ax    = axes[idx]
#         resid = y_test - te_pred[name]

#         s_valid = [s for s in season_order
#                    if s in test_season_str
#                    and (test_season_str == s).sum() > 1]

#         bp_arr = np.empty(len(s_valid), dtype=object)
#         for k, s in enumerate(s_valid):
#             bp_arr[k] = resid[test_season_str == s]

#         parts = ax.boxplot(
#             bp_arr,
#             positions=list(range(len(s_valid))),
#             widths=0.55, patch_artist=True,
#             showfliers=True,
#             flierprops=dict(marker="o", ms=2.5, alpha=0.25),
#             medianprops=dict(color="black", lw=2.0),
#             whiskerprops=dict(lw=1.2),
#             capprops=dict(lw=1.2),
#         )
#         for patch, s in zip(parts["boxes"], s_valid):
#             patch.set_facecolor(SEASON_PAL.get(s, "#888"))
#             patch.set_alpha(0.80)

#         ax.axhline(0, color="red", lw=1.6, ls="--", label="Zero error")
#         ax.set_xticks(range(len(s_valid)))
#         ax.set_xticklabels(s_valid, rotation=15, ha="right")
#         ax.set_ylabel("Residual (Actual-Pred) ug/m3")
#         ax.set_title(name, fontweight="bold", color="#1D3557")
#         ax.legend(fontsize=8)
#         for ki, arr in enumerate(bp_arr):
#             ax.text(ki, float(np.percentile(arr, 92)) + 1.5,
#                     f"m={arr.mean():.1f}", ha="center", fontsize=7.5, color="#333")

#     for j in range(idx+1, len(axes)):
#         axes[j].set_visible(False)

#     plt.tight_layout()
#     save(fig, "plot06_residuals_by_season.png")
# else:
#     print("  'season' not available — skipping plot 06")


# ═══════════════════════════════════════════════════════════
# PLOT 07 — Feature Importances
# ═══════════════════════════════════════════════════════════
print("[Plot 07] Feature importances ...")

base_models = {n: m for n, m in models.items()
               if hasattr(m, "feature_importances_")}

if base_models:
    n_bm  = len(base_models)
    fig, axes = plt.subplots(1, n_bm, figsize=(7*n_bm, 10))
    if n_bm == 1:
        axes = [axes]
    fig.suptitle("Plot 07 — Top-20 Feature Importances",
                 fontsize=14, fontweight="bold", color="#1D3557", y=1.01)

    for ax, (name, m) in zip(axes, base_models.items()):
        fi = pd.Series(m.feature_importances_, index=feature_cols)\
               .sort_values(ascending=False).head(20)
        col  = C.get(name, "#888")
        bars = ax.barh(range(len(fi)), fi.values[::-1],
                       color=col, alpha=0.82, edgecolor="white")
        ax.set_yticks(range(len(fi)))
        ax.set_yticklabels(fi.index[::-1], fontsize=8)
        ax.set_xlabel("Importance")
        ax.set_title(name, fontweight="bold", color="#1D3557")
        for bar, v in zip(bars, fi.values[::-1]):
            ax.text(v + fi.values.max() * 0.01,
                    bar.get_y() + bar.get_height() / 2,
                    f"{v:.4f}", va="center", fontsize=7)

    plt.tight_layout()
    save(fig, "plot07_feature_importances.png")


# ═══════════════════════════════════════════════════════════
# PLOT 08 — Feature Correlation Heatmap
# ═══════════════════════════════════════════════════════════
print("[Plot 08] Feature correlation heatmap ...")

tmp_tr = X_train.copy()
tmp_tr["pm25"] = y_train
corr_with_target = (
    tmp_tr.corr()["pm25"]
    .drop("pm25")
    .abs()
    .sort_values(ascending=False)
    .head(25)
)
top_feats = corr_with_target.index.tolist()
corr_mat  = X_train[top_feats].corr()

fig, ax = plt.subplots(figsize=(14, 12))
fig.suptitle("Plot 08 — Feature Correlation Heatmap (Top-25 by |r| with PM2.5)",
             fontsize=13, fontweight="bold", color="#1D3557")

sns.heatmap(corr_mat,
            mask=np.triu(np.ones_like(corr_mat, dtype=bool)),
            ax=ax, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=0.4, linecolor="white",
            annot=True, fmt=".2f", annot_kws={"size": 6.5},
            cbar_kws={"shrink": 0.75})
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right", fontsize=7.5)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=7.5)
plt.tight_layout()
save(fig, "plot08_feature_correlation_heatmap.png")


# ═══════════════════════════════════════════════════════════
# PLOT 09 — PM2.5 vs Key Features
# ═══════════════════════════════════════════════════════════
print("[Plot 09] PM2.5 vs key features ...")

key_feats = [f for f in [
    "AOD_mean", "PBLH_mean", "SPEED_mean", "moisture_mean",
    "vent_coeff_mean", "inversion_proxy_mean", "PM_proxy", "trap_score",
] if f in X_train.columns][:8]

ncols_f = 4
nrows_f = (len(key_feats) + ncols_f - 1) // ncols_f
fig, axes = plt.subplots(nrows_f, ncols_f, figsize=(6*ncols_f, 5*nrows_f))
axes = np.array(axes).flatten()
fig.suptitle("Plot 09 — PM2.5 vs Key Features (Training Data)",
             fontsize=13, fontweight="bold", color="#1D3557", y=1.01)

for i, feat in enumerate(key_feats):
    ax  = axes[i]
    tmp = pd.DataFrame({feat: X_train[feat].values, "pm25": y_train})
    tmp = tmp.replace(-1, np.nan).dropna()
    if len(tmp) < 50:
        ax.set_visible(False)
        continue

    ax.scatter(tmp[feat], tmp["pm25"],
               alpha=0.12, s=5, color="#457B9D", rasterized=True)
    sl, ic, r, *_ = stats.linregress(tmp[feat], tmp["pm25"])
    xs = np.linspace(float(tmp[feat].min()), float(tmp[feat].max()), 200)
    ax.plot(xs, sl*xs + ic, color="#E63946", lw=2.2, label=f"r = {r:.3f}")
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel("PM2.5 (ug/m3)", fontsize=9)
    ax.set_title(feat, fontweight="bold", color="#1D3557")
    ax.legend(fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
save(fig, "plot09_pm25_vs_features.png")


# ═══════════════════════════════════════════════════════════
# PLOT 10 — Summary Table + Ensemble Weights
# ═══════════════════════════════════════════════════════════
print("[Plot 10] Summary table + ensemble weights ...")

fig = plt.figure(figsize=(20, 9))
fig.suptitle("Plot 10 — Full Metrics Summary & Ensemble Composition",
             fontsize=14, fontweight="bold", color="#1D3557", y=1.01)
gs     = gridspec.GridSpec(1, 2, figure=fig, wspace=0.40)
ax_tbl = fig.add_subplot(gs[0, 0])
ax_pie = fig.add_subplot(gs[0, 1])

cols_t = ["Model","Train R2","Test R2","Delta R2",
          "Train MAE","Test MAE","RMSE","Bias"]
rows   = []
for n in model_names:
    tr, te = tr_metrics[n], te_metrics[n]
    gap    = tr["R2"] - te["R2"]
    rows.append([
        n + (" (!)" if gap > 0.08 else ""),
        f"{tr['R2']:.4f}", f"{te['R2']:.4f}", f"{gap:.4f}",
        f"{tr['MAE']:.2f}", f"{te['MAE']:.2f}",
        f"{te['RMSE']:.2f}", f"{te['Bias']:.2f}",
    ])

ax_tbl.axis("off")
tbl = ax_tbl.table(cellText=rows, colLabels=cols_t,
                   cellLoc="center", loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 2.2)
for j in range(len(cols_t)):
    tbl[0, j].set_facecolor("#1D3557")
    tbl[0, j].set_text_props(color="white", fontweight="bold")
for i, (row, name) in enumerate(zip(rows, model_names)):
    gap = tr_metrics[name]["R2"] - te_metrics[name]["R2"]
    bg  = "#FDECEA" if gap > 0.08 else ("#F0F4F8" if i % 2 == 0 else "white")
    for j in range(len(cols_t)):
        tbl[i+1, j].set_facecolor(bg)

ax_tbl.set_title("Model Performance Summary  (!) = R2 gap > 0.08",
                  fontweight="bold", color="#1D3557", pad=12)

if ens_weights:
    key_map2 = {"rf":"RandomForest","xgb":"XGBoost",
                "lgb":"LightGBM","gb":"GradientBoosting"}
    labels_e = [key_map2.get(k, k) for k in ens_weights]
    vals_e   = list(ens_weights.values())
    colors_e = [C.get(l, "#888") for l in labels_e]
    _, _, autotexts = ax_pie.pie(
        vals_e, labels=labels_e, colors=colors_e,
        autopct="%1.1f%%", startangle=140, pctdistance=0.75,
        wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2),
    )
    for at in autotexts:
        at.set_fontsize(9)
    ax_pie.set_title("Ensemble Blend Weights (weighted by Test R2)",
                      fontweight="bold", color="#1D3557", pad=12)
else:
    ax_pie.text(0.5, 0.5, "ensemble_weights.pkl not found",
                ha="center", va="center", transform=ax_pie.transAxes,
                color="gray", fontsize=11)

plt.tight_layout()
save(fig, "plot10_summary_table_ensemble.png")


# ═══════════════════════════════════════════════════════════
# PLOT 11 — Metric Bar Chart Overview
# ═══════════════════════════════════════════════════════════
print("[Plot 11] Metric bar chart overview ...")

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
fig.suptitle("Plot 11 — Test Set Metric Overview (All Models)",
             fontsize=14, fontweight="bold", color="#1D3557", y=1.02)

for ax, met in zip(axes, ["R2", "MAE", "RMSE", "Bias"]):
    vals  = [te_metrics[n][met] for n in model_names]
    cols  = [C.get(n, "#888")   for n in model_names]
    bars  = ax.bar(range(len(model_names)), vals,
                   color=cols, alpha=0.85, edgecolor="white", width=0.55)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, rotation=22, ha="right")
    ax.set_ylabel(met)
    ax.set_title(met, fontweight="bold", color="#1D3557")
    ax.axhline(0, color="black", lw=0.6, alpha=0.2)
    if met == "R2":
        ax.axhline(1.0, color="black", lw=0.8, ls=":", alpha=0.4)
    rng = max(vals) - min(vals) if max(vals) != min(vals) else 1
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + rng * 0.03,
                f"{v:.3f}", ha="center", fontsize=8, fontweight="bold")

plt.tight_layout()
save(fig, "plot11_metric_overview.png")


# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print(f"  All 11 plots saved to:")
print(f"  {REPORT_DIR}")
print("=" * 60)
print("""
  plot01 — Overfitting diagnosis  (Train vs Test R2/MAE/RMSE)
  plot02 — Predicted vs Actual scatter  (per model)
  plot03 — Time-series: Actual vs all models  (500 samples)
  plot04 — Residual distributions with KDE
  plot05 — MAE by PM2.5 concentration bin
  plot06 — Residuals by season  (per model)
  plot07 — Top-20 feature importances  (per model)
  plot08 — Feature correlation heatmap  (top-25)
  plot09 — PM2.5 vs key features  (scatter + regression)
  plot10 — Full metrics table + ensemble weights
  plot11 — Clean metric bar chart overview
""")

  Loading data ...
  Shape : (66617, 63)
  Features : 57
  Train : (49918, 57)  |  Test : (16699, 57)

  Loading models ...
    Loaded : RandomForest
    Loaded : XGBoost
    Loaded : LightGBM
    Loaded : GradientBoosting
    Column order aligned  (57 features)
    Ensemble predictions computed

  OVERFITTING CHECK
  Model                   Train R2   Test R2   Delta R2   Train MAE  Test MAE  Delta MAE
  -------------------------------------------------------------------------------------
  RandomForest              0.8850    0.6924     0.1925        9.17     15.08       5.91  << OVERFIT
  XGBoost                   0.8020    0.7012     0.1009       12.74     14.97       2.23  << OVERFIT
  LightGBM                  0.7983    0.7049     0.0934       12.83     14.89       2.07  << OVERFIT
  GradientBoosting          0.8564    0.6913     0.1652       10.79     15.13       4.33  << OVERFIT
  Ensemble                  0.8420    0.7053     0.1367       11.23     14.84       3.60  << OVERFIT
